# 조건별 평가와 오류 분류

전체 30개 질문을 읽고 dev 15개로 설정을 검토합니다.

## 실행 환경 준비

이 노트북은 제공 PDF에서 실제 텍스트를 추출합니다. 답변 생성 API는 호출하지 않습니다.

In [1]:
from pathlib import Path
import json, math, re
from collections import Counter
from pypdf import PdfReader

# 현재 폴더에서 저장소의 공개 RAG 자료를 찾습니다.
root = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "labs/data/rag-corpus/manifest.json").exists())
corpus = root / "labs/data/rag-corpus"
dataset_path = root / "labs/part3/ch11-evaluation/data/evaluation-queries.jsonl"
dataset = [json.loads(line) for line in dataset_path.read_text(encoding="utf-8").splitlines()]
evaluation_items = [item for item in dataset if item["split"] == "dev"]
print(f"전체 질문: {len(dataset)}개")
print(f"설정 선택용 dev 질문: {len(evaluation_items)}개")

전체 질문: 30개
설정 선택용 dev 질문: 15개


## 공개 PDF를 페이지 단위 후보로 준비

스캔본은 OCR이 필요하므로 이번 기초 검색에서 제외합니다.

In [2]:
manifest = json.loads((corpus / "manifest.json").read_text(encoding="utf-8"))
pages = []
for item in manifest:
    if item["tier"] == "scanned":
        continue
    reader = PdfReader(corpus / item["file"])
    for page_no, page in enumerate(reader.pages, start=1):
        text = page.extract_text() or ""
        if text.strip():
            pages.append({"doc_id": item["doc_id"], "page": page_no, "text": text})
print(f"검색 후보: {len(pages)}개 페이지")

검색 후보: 90개 페이지


## 같은 검색기를 모든 질문에 적용

단어와 한글 두 글자 조각의 겹침을 사용합니다. 점수가 같으면 문서 ID와 쪽 번호로 순서를 고정합니다.

In [3]:
def tokens(text):
    normalized = re.sub(r"\s+", " ", text.lower())
    words = re.findall(r"[가-힣a-z0-9]+", normalized)
    compact = re.sub(r"[^가-힣a-z0-9]", "", normalized)
    return words + [compact[i:i+2] for i in range(max(0, len(compact)-1))]

def search(question, limit=50):
    query_counts = Counter(tokens(question))
    scored = []
    for page in pages:
        page_counts = Counter(tokens(page["text"]))
        overlap = sum(min(count, page_counts[token]) for token, count in query_counts.items())
        score = overlap / math.sqrt(max(1, sum(page_counts.values())))
        scored.append({"doc_id": page["doc_id"], "page": page["page"], "score": score})
    ranked_pages = sorted(scored, key=lambda row: (-row["score"], row["doc_id"], row["page"]))
    unique_docs, seen = [], set()
    for row in ranked_pages:
        if row["doc_id"] not in seen:
            unique_docs.append(row)
            seen.add(row["doc_id"])
    return unique_docs[:limit]

sample = search(dataset[0]["question"], 5)
print([(row["doc_id"], row["page"], round(row["score"], 3)) for row in sample])

[('E01', 1, 0.926), ('M04', 3, 0.225), ('H03', 1, 0.188), ('H06', 5, 0.142), ('E02', 5, 0.132)]


## 질문 유형별로 나누어 측정

전체 평균과 유형별 평균을 함께 봅니다.

In [4]:
eligible = [item for item in evaluation_items if item["answerable"] and item["gold_doc_ids"]]
excluded_count = len(evaluation_items) - len(eligible)

def metrics(item):
    results = search(item["question"], 50)
    gold = set(item["gold_doc_ids"])
    r50 = len({row["doc_id"] for row in results} & gold) / len(gold) if gold else 0.0
    p5 = sum(row["doc_id"] in gold for row in results[:5]) / 5
    return r50, p5

grouped = {}
for item in eligible:
    grouped.setdefault(item["type"], []).append(metrics(item))
for name, values in sorted(grouped.items()):
    print(name, len(values), f"R@50={sum(v[0] for v in values)/len(values):.3f}",
          f"P@5={sum(v[1] for v in values)/len(values):.3f}")
print("검색 평균 제외:", excluded_count)

multi_doc 1 R@50=1.000 P@5=0.200
multi_hop 1 R@50=1.000 P@5=0.200
reasoning 1 R@50=1.000 P@5=0.200
single 9 R@50=1.000 P@5=0.200
table 2 R@50=1.000 P@5=0.200
temporal 1 R@50=1.000 P@5=0.200
검색 평균 제외: 0


## 오류를 검색 단계 이름으로 분류

후보 누락과 상위 순위 문제를 서로 다른 이름으로 기록합니다.

In [5]:
labels = Counter()
for item in eligible:
    results = search(item["question"], 50)
    gold = set(item["gold_doc_ids"])
    top50 = {row["doc_id"] for row in results}
    top5 = {row["doc_id"] for row in results[:5]}
    if not top50 & gold:
        labels["candidate_miss"] += 1
    elif not top5 & gold:
        labels["ranking_miss"] += 1
    else:
        labels["top5_hit"] += 1
print(dict(labels))

{'top5_hit': 15}
